# Ozon Counterfeit: Improvement Experiments

Self-contained notebook for testing four targeted multimodal improvements:

1. Stronger text embeddings via `intfloat/multilingual-e5-base`.
2. Image-text consistency features through SentenceTransformer CLIP embeddings.
3. Tabular cleanup by excluding technical `row_id` from numeric features.
4. Small-grid tuning for gated multimodal fusion.

This notebook does not modify existing notebooks, `src/*`, or `scripts/*`.


## Configuration

In [1]:
import os
from pathlib import Path

os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

PROJECT_ROOT = Path.cwd().resolve()

TRAIN_CSV = PROJECT_ROOT / 'data/ml_ozon_сounterfeit_data/ml_ozon_сounterfeit_train.csv'
TRAIN_IMAGES_DIR = PROJECT_ROOT / 'data/ml_ozon_сounterfeit_train_images'
MANIFEST_PATH = PROJECT_ROOT / 'artifacts/manifests/train_manifest.csv'

EXPERIMENT_NAMESPACE = 'all_modalities_v2'
TEXT_MODEL_NAME = 'intfloat/multilingual-e5-base'
IMAGE_MODEL_NAME = 'efficientnet_b0'
CLIP_MODEL_NAME = 'clip-ViT-B-32'
SELECTION_METRIC = 'pr_auc'

# First full run: True. Later reuse run: False.
FORCE_RETRAIN = True

# Smoke run: set to 1000. Full run: None.
MAX_SAMPLES_PER_SPLIT = None

RUN_CLIP_FEATURES = True
RUN_SIMPLE_MODELS = True
RUN_GATED_GRID = True
RUN_LATE_FUSION = True

DEVICE = 'mps'
BATCH_SIZE = 256
IMAGE_BATCH_SIZE = max(8, BATCH_SIZE // 4)
CLIP_BATCH_SIZE = 128
NUM_WORKERS = 0

TECHNICAL_NUMERIC_COLUMNS = {'row_id'}
CLIP_CONSISTENCY_COLUMNS = [
    'clip_image_text_cosine',
    'clip_image_brand_name_cosine',
    'clip_image_name_rus_cosine',
    'clip_image_category_cosine',
]

LOGISTIC_MAX_ITER = 2000
LOGISTIC_C = 1.0
LATE_FUSION_ALPHAS = None  # Initialized after numpy import.

ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
NAMESPACE_DIR = ARTIFACTS_DIR / 'experiments' / EXPERIMENT_NAMESPACE
CACHE_DIR = NAMESPACE_DIR / 'cache'
REPORTS_DIR = ARTIFACTS_DIR / 'reports'
RESULTS_CSV_PATH = REPORTS_DIR / f'{EXPERIMENT_NAMESPACE}_results.csv'
SPLIT_CHECKS_PATH = REPORTS_DIR / f'{EXPERIMENT_NAMESPACE}_split_checks.json'
V1_RESULTS_PATH = REPORTS_DIR / 'all_modalities_efficientnet_b0_results.csv'
V1_COMPARISON_PATH = REPORTS_DIR / f'{EXPERIMENT_NAMESPACE}_vs_v1.csv'

PROJECT_ROOT


PosixPath('/Users/kovalev.fedor2005yandex.ru/Desktop/Диплом/Кодовая_база')

## Imports

In [2]:
if False:
    %pip install -r requirements.txt


In [3]:
import csv
import pickle
import sys
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from ozon_counterfeit.constants import RANDOM_SEED
from ozon_counterfeit.data import load_train_frame, merge_manifest_with_frame
from ozon_counterfeit.encoders import FrozenImageEncoder, FrozenTextEncoder, load_cached_array, save_cached_array
from ozon_counterfeit.features import (
    TabularPreprocessor,
    build_modality_flags,
    build_text_corpus,
    extract_labels,
    infer_log_columns,
    infer_numeric_columns,
)
from ozon_counterfeit.splitting import compute_split_summary
from ozon_counterfeit.training import NeuralTrainingConfig, choose_threshold_by_f1, compute_metrics
from ozon_counterfeit.utils import ensure_dir, load_json, resolve_torch_device, save_json, set_seed, slugify

LATE_FUSION_ALPHAS = np.linspace(0.0, 1.0, 101)
set_seed(RANDOM_SEED)
print('Imports ready')


Imports ready


## Data Load And Split Checks

In [4]:
def sample_per_split(frame: pd.DataFrame, limit: int) -> pd.DataFrame:
    sampled = []
    for _, split_frame in frame.groupby('split', sort=False):
        take = min(len(split_frame), limit)
        sampled.append(split_frame.sample(n=take, random_state=RANDOM_SEED))
    return pd.concat(sampled, axis=0).sort_values(['split', 'row_id']).reset_index(drop=True)


def load_image_covered_dataset() -> tuple[pd.DataFrame, dict[str, pd.DataFrame], dict[str, object]]:
    train_frame = load_train_frame(TRAIN_CSV)
    manifest_raw = pd.read_csv(MANIFEST_PATH, dtype={'ItemID': str, 'SellerID': str})
    manifest_full = train_frame[['row_id', 'ItemID', 'SellerID', 'resolution']].merge(
        manifest_raw,
        on=['row_id', 'ItemID', 'SellerID', 'resolution'],
        how='inner',
    )
    dataset_full = merge_manifest_with_frame(train_frame, manifest_raw)

    checks: dict[str, object] = {}
    checks['train_rows'] = int(len(train_frame))
    checks['manifest_rows'] = int(len(manifest_full))
    checks['dataset_rows'] = int(len(dataset_full))
    checks['row_count_ok'] = bool(len(manifest_full) == len(train_frame) == len(dataset_full))

    image_rows = dataset_full.loc[dataset_full['has_image'] == 1].copy()
    checks['image_covered_rows'] = int(len(image_rows))
    checks['rows_without_image'] = int((manifest_full['has_image'] == 0).sum())
    checks['all_selected_rows_have_image'] = bool(image_rows['has_image'].eq(1).all())

    exact_mapping_ok = image_rows['image_path'].map(lambda value: Path(value).stem).eq(image_rows['ItemID'].astype(str)).all()
    checks['exact_itemid_image_path_mapping_ok'] = bool(exact_mapping_ok)

    if MAX_SAMPLES_PER_SPLIT:
        image_rows = sample_per_split(image_rows, MAX_SAMPLES_PER_SPLIT)
        checks['max_samples_per_split'] = int(MAX_SAMPLES_PER_SPLIT)
    else:
        checks['max_samples_per_split'] = None

    split_frames = {
        split_name: split_frame.reset_index(drop=True)
        for split_name, split_frame in image_rows.groupby('split', sort=False)
    }
    required_splits = {'train', 'val', 'test'}
    missing_splits = required_splits - set(split_frames)
    checks['required_splits_present'] = bool(not missing_splits)
    checks['missing_splits'] = sorted(missing_splits)
    if missing_splits:
        raise ValueError(f'Missing splits: {sorted(missing_splits)}')

    split_seller_sets = {
        split_name: set(split_frame['SellerID'].astype(str))
        for split_name, split_frame in split_frames.items()
    }
    seller_overlap_counts = {}
    split_names = sorted(split_seller_sets)
    for index, left_name in enumerate(split_names):
        for right_name in split_names[index + 1:]:
            seller_overlap_counts[f'{left_name}__{right_name}'] = len(split_seller_sets[left_name] & split_seller_sets[right_name])
    checks['seller_overlap_counts'] = seller_overlap_counts
    checks['seller_leakage_ok'] = bool(all(count == 0 for count in seller_overlap_counts.values()))

    checks['split_summary'] = compute_split_summary(image_rows).to_dict(orient='records')
    checks['class_balance'] = {
        split_name: {
            'rows': int(len(split_frame)),
            'positives': int(split_frame['resolution'].sum()),
            'positive_rate': float(split_frame['resolution'].mean()),
        }
        for split_name, split_frame in split_frames.items()
    }

    critical_checks = [
        checks['row_count_ok'],
        checks['all_selected_rows_have_image'],
        checks['exact_itemid_image_path_mapping_ok'],
        checks['required_splits_present'],
        checks['seller_leakage_ok'],
    ]
    checks['all_critical_checks_ok'] = bool(all(critical_checks))
    if not checks['all_critical_checks_ok']:
        raise ValueError(f'Critical split/data checks failed: {checks}')

    ensure_dir(SPLIT_CHECKS_PATH.parent)
    save_json(SPLIT_CHECKS_PATH, checks)
    return image_rows.reset_index(drop=True), split_frames, checks


dataset, split_frames, split_checks = load_image_covered_dataset()
labels = {split: extract_labels(split_frames[split]) for split in ('train', 'val', 'test')}
display(pd.DataFrame(split_checks['split_summary']))
print(f'Split checks saved to: {SPLIT_CHECKS_PATH}')


,split,rows,positives,negatives,positive_rate,unique_sellers,with_image_rows
0,test,32563,1957,30606,0.060099,60,32563
1,train,132014,9097,122917,0.068909,11060,132014
2,val,31883,1903,29980,0.059687,34,31883


Split checks saved to: /Users/kovalev.fedor2005yandex.ru/Desktop/Диплом/Кодовая_база/artifacts/reports/all_modalities_v2_split_checks.json


## Shared IO, Metrics, And Cache Helpers

In [5]:
def save_pickle(path: str | Path, payload: object) -> None:
    path = Path(path)
    ensure_dir(path.parent)
    with path.open('wb') as handle:
        pickle.dump(payload, handle)


def load_pickle_local(path: str | Path) -> object:
    with Path(path).open('rb') as handle:
        return pickle.load(handle)


def ensure_finite(name: str, array: np.ndarray) -> None:
    if not np.isfinite(array).all():
        raise ValueError(f'{name} contains NaN or inf values.')


def cache_feature_path(name: str, split: str) -> Path:
    return CACHE_DIR / f'{name}_{split}.npy'


def load_cached_feature(cache_path: Path, expected_rows: int) -> np.ndarray | None:
    if FORCE_RETRAIN:
        return None
    cached = load_cached_array(cache_path)
    if cached is None:
        return None
    if len(cached) != expected_rows:
        print(f'Ignoring stale cache {cache_path}: {len(cached)} rows, expected {expected_rows}')
        return None
    ensure_finite(str(cache_path), cached)
    return cached


def save_feature_dict(name: str, features: dict[str, np.ndarray]) -> None:
    for split, array in features.items():
        ensure_finite(f'{name}_{split}', array)
        save_cached_array(cache_feature_path(name, split), array)


def load_feature_dict(name: str, split_frames: dict[str, pd.DataFrame]) -> dict[str, np.ndarray] | None:
    features = {}
    for split in ('train', 'val', 'test'):
        cached = load_cached_feature(cache_feature_path(name, split), len(split_frames[split]))
        if cached is None:
            return None
        features[split] = cached
    return features


def experiment_dir(name: str) -> Path:
    return NAMESPACE_DIR / name


def has_complete_experiment(name: str, model_filename: str | None = None) -> bool:
    base = experiment_dir(name)
    required = [base / 'metadata.json', base / 'metrics.json', base / 'scores_val_test.npz']
    if model_filename:
        required.append(base / model_filename)
    return all(path.exists() for path in required)


def save_scores(name: str, val_scores: np.ndarray, test_scores: np.ndarray) -> None:
    base = ensure_dir(experiment_dir(name))
    val_scores = val_scores.astype(np.float32)
    test_scores = test_scores.astype(np.float32)
    ensure_finite(f'{name}_val_scores', val_scores)
    ensure_finite(f'{name}_test_scores', test_scores)
    if len(val_scores) != len(labels['val']) or len(test_scores) != len(labels['test']):
        raise ValueError(f'{name} score length mismatch.')
    np.savez_compressed(
        base / 'scores_val_test.npz',
        y_val=labels['val'].astype(np.float32),
        y_test=labels['test'].astype(np.float32),
        val_scores=val_scores,
        test_scores=test_scores,
    )


def load_scores(name: str) -> dict[str, np.ndarray]:
    with np.load(experiment_dir(name) / 'scores_val_test.npz') as payload:
        return {key: payload[key] for key in payload.files}


def metrics_for_scores(val_scores: np.ndarray, test_scores: np.ndarray) -> dict[str, object]:
    threshold = choose_threshold_by_f1(labels['val'], val_scores)
    return {
        'validation': compute_metrics(labels['val'], val_scores, threshold),
        'test': compute_metrics(labels['test'], test_scores, threshold),
    }


def save_experiment(name: str, metadata: dict[str, object], metrics: dict[str, object]) -> None:
    base = ensure_dir(experiment_dir(name))
    metadata = {
        **metadata,
        'experiment': name,
        'namespace': EXPERIMENT_NAMESPACE,
        'text_model_name': TEXT_MODEL_NAME,
        'image_model_name': IMAGE_MODEL_NAME,
        'clip_model_name': CLIP_MODEL_NAME,
        'selection_metric': SELECTION_METRIC,
        'max_samples_per_split': MAX_SAMPLES_PER_SPLIT,
        'train_rows': int(len(split_frames['train'])),
        'val_rows': int(len(split_frames['val'])),
        'headline_eval_rows': int(len(split_frames['test'])),
    }
    save_json(base / 'metadata.json', metadata)
    save_json(base / 'metrics.json', metrics)


def load_experiment_metrics(name: str) -> dict[str, object]:
    return load_json(experiment_dir(name) / 'metrics.json')


def compact_metric_row(name: str, group: str, metrics: dict[str, object]) -> dict[str, object]:
    val = metrics['validation']
    test = metrics['test']
    return {
        'experiment': name,
        'group': group,
        'val_pr_auc': val['pr_auc'],
        'val_precision': val['precision'],
        'val_recall': val['recall'],
        'val_f1': val['f1'],
        'test_pr_auc': test['pr_auc'],
        'test_precision': test['precision'],
        'test_recall': test['recall'],
        'test_f1': test['f1'],
        'threshold': test['threshold'],
    }


def validate_score_alignment(experiment_names: list[str]) -> dict[str, object]:
    report = {'experiments': experiment_names, 'aligned': True, 'details': {}}
    reference = None
    for name in experiment_names:
        payload = load_scores(name)
        detail = {
            'val_len': int(len(payload['val_scores'])),
            'test_len': int(len(payload['test_scores'])),
            'val_finite': bool(np.isfinite(payload['val_scores']).all()),
            'test_finite': bool(np.isfinite(payload['test_scores']).all()),
        }
        current = (payload['y_val'], payload['y_test'])
        if reference is None:
            reference = current
            detail['label_alignment_ok'] = True
        else:
            detail['label_alignment_ok'] = bool(np.array_equal(reference[0], current[0]) and np.array_equal(reference[1], current[1]))
        if not all([detail['val_finite'], detail['test_finite'], detail['label_alignment_ok']]):
            report['aligned'] = False
        report['details'][name] = detail
    return report


ensure_dir(CACHE_DIR)


PosixPath('/Users/kovalev.fedor2005yandex.ru/Desktop/Диплом/Кодовая_база/artifacts/experiments/all_modalities_v2/cache')

## Feature Cache: Tabular Without row_id

In [6]:
def build_or_load_tabular_features_without_row_id(split_frames: dict[str, pd.DataFrame]) -> dict[str, np.ndarray]:
    cached = load_feature_dict('tabular_without_row_id', split_frames)
    metadata_path = CACHE_DIR / 'tabular_without_row_id_metadata.json'
    preprocessor_path = CACHE_DIR / 'tabular_without_row_id_preprocessor.pkl'
    if cached is not None and metadata_path.exists() and preprocessor_path.exists():
        return cached

    all_numeric_columns = infer_numeric_columns(load_train_frame(TRAIN_CSV))
    numeric_columns = [column for column in all_numeric_columns if column not in TECHNICAL_NUMERIC_COLUMNS]
    excluded_columns = [column for column in all_numeric_columns if column in TECHNICAL_NUMERIC_COLUMNS]
    log_columns = infer_log_columns(numeric_columns)
    preprocessor = TabularPreprocessor(numeric_columns=numeric_columns, log_columns=log_columns)
    features = {
        'train': preprocessor.fit_transform(split_frames['train']),
        'val': preprocessor.transform(split_frames['val']),
        'test': preprocessor.transform(split_frames['test']),
    }
    save_feature_dict('tabular_without_row_id', features)
    save_pickle(preprocessor_path, preprocessor)
    save_json(
        metadata_path,
        {
            'numeric_columns': numeric_columns,
            'log_columns': log_columns,
            'excluded_technical_columns': excluded_columns,
            'feature_dim': int(features['train'].shape[1]),
        },
    )
    return features


tabular_features = build_or_load_tabular_features_without_row_id(split_frames)
display(pd.DataFrame([{'split': split, 'shape': tabular_features[split].shape} for split in ('train', 'val', 'test')]))


,split,shape
0,train,"(132014, 74)"
1,val,"(31883, 74)"
2,test,"(32563, 74)"


## Feature Cache: E5 Text Embeddings

In [8]:
def build_e5_corpus(frame: pd.DataFrame) -> list[str]:
    return [f'passage: {text}' for text in build_text_corpus(frame)]


def build_or_load_e5_text_features(split_frames: dict[str, pd.DataFrame]) -> dict[str, np.ndarray]:
    feature_name = f'text_e5_{slugify(TEXT_MODEL_NAME)}'
    cached = load_feature_dict(feature_name, split_frames)
    if cached is not None:
        return cached

    encoder = FrozenTextEncoder(TEXT_MODEL_NAME, device=DEVICE)
    features = {}
    for split in ('train', 'val', 'test'):
        texts = build_e5_corpus(split_frames[split])
        features[split] = encoder.encode(texts, batch_size=16)
        ensure_finite(f'{feature_name}_{split}', features[split])
    save_feature_dict(feature_name, features)
    return features


text_features = build_or_load_e5_text_features(split_frames)
display(pd.DataFrame([{'split': split, 'shape': text_features[split].shape} for split in ('train', 'val', 'test')]))


/Users/kovalev.fedor2005yandex.ru/Desktop/Диплом/Кодовая_база/src/ozon_counterfeit/encoders.py:37: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.embedding_dim = self.model.get_sentence_embedding_dimension()


Batches:   0%|          | 0/8251 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Feature Cache: EfficientNet Image Embeddings

In [ ]:
def build_or_load_image_features(split_frames: dict[str, pd.DataFrame]) -> dict[str, np.ndarray]:
    feature_name = f'image_{slugify(IMAGE_MODEL_NAME)}'
    cached = load_feature_dict(feature_name, split_frames)
    if cached is not None:
        return cached

    encoder = FrozenImageEncoder(IMAGE_MODEL_NAME, device=DEVICE)
    features = {}
    for split in ('train', 'val', 'test'):
        image_paths = split_frames[split]['image_path'].astype(str).tolist()
        features[split] = encoder.encode(image_paths, batch_size=IMAGE_BATCH_SIZE, num_workers=NUM_WORKERS)
        ensure_finite(f'{feature_name}_{split}', features[split])
    save_feature_dict(feature_name, features)
    return features


image_features = build_or_load_image_features(split_frames)
display(pd.DataFrame([{'split': split, 'shape': image_features[split].shape} for split in ('train', 'val', 'test')]))


## Feature Cache: CLIP/SentenceTransformer Consistency Features

In [ ]:
def normalize_prompt_values(values: pd.Series, fallback: str) -> list[str]:
    normalized = values.fillna('').astype(str).str.strip()
    normalized = normalized.mask(normalized.eq(''), fallback)
    return normalized.tolist()


def clip_text_sets(frame: pd.DataFrame) -> dict[str, list[str]]:
    return {
        'full_text': build_text_corpus(frame),
        'brand_name': [f'brand: {value}' for value in normalize_prompt_values(frame['brand_name'], 'unknown brand')],
        'name_rus': [f'product name: {value}' for value in normalize_prompt_values(frame['name_rus'], 'unknown product')],
        'category': [f'category: {value}' for value in normalize_prompt_values(frame['CommercialTypeName4'], 'unknown category')],
    }


def encode_clip_images(model, image_paths: list[str], cache_path: Path) -> np.ndarray:
    cached = load_cached_feature(cache_path, len(image_paths))
    if cached is not None:
        return cached

    embeddings = []
    iterator = range(0, len(image_paths), CLIP_BATCH_SIZE)
    if tqdm is not None:
        iterator = tqdm(iterator, desc=f'CLIP images {cache_path.stem}')
    for start in iterator:
        batch_paths = image_paths[start:start + CLIP_BATCH_SIZE]
        images = []
        for path in batch_paths:
            with Image.open(path) as image:
                images.append(image.convert('RGB'))
        batch_embeddings = model.encode(
            images,
            batch_size=CLIP_BATCH_SIZE,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        embeddings.append(batch_embeddings.astype(np.float32))
    array = np.concatenate(embeddings, axis=0).astype(np.float32)
    ensure_finite(str(cache_path), array)
    return save_cached_array(cache_path, array)


def encode_clip_texts(model, texts: list[str], cache_path: Path) -> np.ndarray:
    cached = load_cached_feature(cache_path, len(texts))
    if cached is not None:
        return cached
    array = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)
    ensure_finite(str(cache_path), array)
    return save_cached_array(cache_path, array)


def cosine_from_normalized(left: np.ndarray, right: np.ndarray) -> np.ndarray:
    return np.clip(np.sum(left * right, axis=1), -1.0, 1.0).reshape(-1, 1).astype(np.float32)


def build_or_load_clip_consistency_features(split_frames: dict[str, pd.DataFrame]) -> dict[str, np.ndarray]:
    if not RUN_CLIP_FEATURES:
        return {split: np.zeros((len(split_frames[split]), len(CLIP_CONSISTENCY_COLUMNS)), dtype=np.float32) for split in ('train', 'val', 'test')}

    feature_name = f'clip_consistency_{slugify(CLIP_MODEL_NAME)}'
    cached = load_feature_dict(feature_name, split_frames)
    if cached is not None:
        return cached

    try:
        from sentence_transformers import SentenceTransformer
        clip_model = SentenceTransformer(CLIP_MODEL_NAME, device=resolve_torch_device(DEVICE))
    except Exception as error:
        raise RuntimeError(
            f'Could not initialize SentenceTransformer CLIP model {CLIP_MODEL_NAME!r}. '
            'Install/verify sentence-transformers CLIP support or set RUN_CLIP_FEATURES=False for a no-CLIP run.'
        ) from error

    features = {}
    for split in ('train', 'val', 'test'):
        split_frame = split_frames[split]
        image_paths = split_frame['image_path'].astype(str).tolist()
        image_embeddings = encode_clip_images(
            clip_model,
            image_paths,
            CACHE_DIR / f'clip_image_{split}_{slugify(CLIP_MODEL_NAME)}.npy',
        )
        text_payloads = clip_text_sets(split_frame)
        cosine_columns = []
        for key in ('full_text', 'brand_name', 'name_rus', 'category'):
            text_embeddings = encode_clip_texts(
                clip_model,
                text_payloads[key],
                CACHE_DIR / f'clip_text_{key}_{split}_{slugify(CLIP_MODEL_NAME)}.npy',
            )
            cosine_columns.append(cosine_from_normalized(image_embeddings, text_embeddings))
        features[split] = np.hstack(cosine_columns).astype(np.float32)
        ensure_finite(f'{feature_name}_{split}', features[split])
    save_feature_dict(feature_name, features)
    save_json(CACHE_DIR / f'{feature_name}_metadata.json', {'columns': CLIP_CONSISTENCY_COLUMNS, 'clip_model_name': CLIP_MODEL_NAME})
    return features


clip_consistency_features = build_or_load_clip_consistency_features(split_frames)
display(pd.DataFrame([{'split': split, 'shape': clip_consistency_features[split].shape} for split in ('train', 'val', 'test')]))


## Simple Models With Improved Features

In [ ]:
def concat_features(parts: list[dict[str, np.ndarray]]) -> dict[str, np.ndarray]:
    return {
        split: np.hstack([part[split] for part in parts]).astype(np.float32)
        for split in ('train', 'val', 'test')
    }


def train_or_load_logistic_experiment(name: str, features: dict[str, np.ndarray], group: str, feature_names: list[str]) -> dict[str, object]:
    if not FORCE_RETRAIN and has_complete_experiment(name, model_filename='model.pkl'):
        return {'name': name, 'metrics': load_experiment_metrics(name), 'metadata': load_json(experiment_dir(name) / 'metadata.json')}

    for split, array in features.items():
        ensure_finite(f'{name}_{split}_features', array)

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=LOGISTIC_MAX_ITER,
            C=LOGISTIC_C,
            class_weight='balanced',
            solver='saga',
            n_jobs=-1,
            random_state=RANDOM_SEED,
        ),
    )
    model.fit(features['train'], labels['train'].astype(int))
    val_scores = model.predict_proba(features['val'])[:, 1].astype(np.float32)
    test_scores = model.predict_proba(features['test'])[:, 1].astype(np.float32)
    metrics = metrics_for_scores(val_scores, test_scores)
    metadata = {
        'group': group,
        'model_type': 'logistic_regression',
        'features': feature_names,
        'feature_dim': int(features['train'].shape[1]),
        'logistic_max_iter': LOGISTIC_MAX_ITER,
        'logistic_c': LOGISTIC_C,
    }
    save_pickle(experiment_dir(name) / 'model.pkl', model)
    save_scores(name, val_scores, test_scores)
    save_experiment(name, metadata, metrics)
    return {'name': name, 'metrics': metrics, 'metadata': metadata}


simple_results = []
if RUN_SIMPLE_MODELS:
    simple_specs = {
        'tabular_text_e5_simple': {
            'features': concat_features([tabular_features, text_features]),
            'feature_names': ['tabular_without_row_id', 'text_e5'],
        },
        'tabular_text_image_e5_clip_features_simple': {
            'features': concat_features([tabular_features, text_features, image_features, clip_consistency_features]),
            'feature_names': ['tabular_without_row_id', 'text_e5', 'image_efficientnet_b0', 'clip_consistency_features'],
        },
    }
    for name, spec in simple_specs.items():
        result = train_or_load_logistic_experiment(name, spec['features'], 'simple_improved', spec['feature_names'])
        simple_results.append(result)
        display(pd.DataFrame([compact_metric_row(name, 'simple_improved', result['metrics'])]))
else:
    print('RUN_SIMPLE_MODELS is disabled')


## Small Grid Gated Fusion Tuning

In [ ]:
class NotebookProjectionBlock(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, dropout: float) -> None:
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(input_dim, output_dim), nn.LayerNorm(output_dim), nn.ReLU(), nn.Dropout(dropout))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.layers(inputs)


class GatedFusionClassifier(nn.Module):
    def __init__(self, tabular_dim: int, text_dim: int, image_dim: int, projection_dim: int, hidden_dim: int, dropout: float) -> None:
        super().__init__()
        self.tabular_projector = NotebookProjectionBlock(tabular_dim, projection_dim, dropout)
        self.text_projector = NotebookProjectionBlock(text_dim, projection_dim, dropout)
        self.image_projector = NotebookProjectionBlock(image_dim, projection_dim, dropout)
        gate_input_dim = projection_dim * 3 + 3
        self.image_gate = nn.Sequential(
            nn.Linear(gate_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, projection_dim),
            nn.Sigmoid(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(projection_dim * 3 + 3, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, tabular_features: torch.Tensor, text_features: torch.Tensor, image_features: torch.Tensor, modality_flags: torch.Tensor) -> torch.Tensor:
        tabular_projection = self.tabular_projector(tabular_features)
        text_projection = self.text_projector(text_features)
        image_projection = self.image_projector(image_features)
        gate_inputs = torch.cat([tabular_projection, text_projection, image_projection, modality_flags], dim=1)
        gated_image_projection = image_projection * self.image_gate(gate_inputs)
        fused = torch.cat([tabular_projection, text_projection, gated_image_projection, modality_flags], dim=1)
        return self.classifier(fused).squeeze(1)


class FusionDataset(Dataset):
    def __init__(self, tabular: np.ndarray, text: np.ndarray, image: np.ndarray, flags: np.ndarray, y: np.ndarray) -> None:
        self.tabular = torch.from_numpy(tabular.astype(np.float32))
        self.text = torch.from_numpy(text.astype(np.float32))
        self.image = torch.from_numpy(image.astype(np.float32))
        self.flags = torch.from_numpy(flags.astype(np.float32))
        self.y = torch.from_numpy(y.astype(np.float32))

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, index: int):
        return self.tabular[index], self.text[index], self.image[index], self.flags[index], self.y[index]


def make_neural_inputs() -> tuple[dict[str, np.ndarray], dict[str, np.ndarray], dict[str, np.ndarray]]:
    tabular_plus_clip = concat_features([tabular_features, clip_consistency_features])
    packed = {}
    for split in ('train', 'val', 'test'):
        packed[split] = {
            'tabular': tabular_plus_clip[split],
            'text': text_features[split],
            'image': image_features[split],
            'flags': build_modality_flags(len(split_frames[split]), np.ones(len(split_frames[split]), dtype=np.float32)),
            'labels': labels[split],
        }
    return packed['train'], packed['val'], packed['test']


def make_loader(inputs: dict[str, np.ndarray], batch_size: int, shuffle: bool) -> DataLoader:
    dataset = FusionDataset(inputs['tabular'], inputs['text'], inputs['image'], inputs['flags'], inputs['labels'])
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=False)


def predict_neural_scores(model: nn.Module, loader: DataLoader, device: str, desc: str) -> np.ndarray:
    model.eval()
    scores = []
    iterator = tqdm(loader, desc=desc, leave=False) if tqdm is not None else loader
    with torch.no_grad():
        for tabular, text, image, flags, _ in iterator:
            logits = model(tabular.to(device), text.to(device), image.to(device), flags.to(device))
            scores.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(scores, axis=0).astype(np.float32)


GATED_GRID = [
    {'config_id': 'cfg_a', 'learning_rate': 1e-3, 'dropout': 0.20, 'weight_decay': 1e-4, 'projection_dim': 256, 'hidden_dim': 256, 'epochs': 12, 'patience': 3},
    {'config_id': 'cfg_b', 'learning_rate': 5e-4, 'dropout': 0.30, 'weight_decay': 3e-4, 'projection_dim': 256, 'hidden_dim': 256, 'epochs': 12, 'patience': 4},
    {'config_id': 'cfg_c', 'learning_rate': 5e-4, 'dropout': 0.25, 'weight_decay': 1e-4, 'projection_dim': 128, 'hidden_dim': 512, 'epochs': 12, 'patience': 4},
    {'config_id': 'cfg_d', 'learning_rate': 1e-3, 'dropout': 0.30, 'weight_decay': 3e-4, 'projection_dim': 128, 'hidden_dim': 256, 'epochs': 12, 'patience': 4},
    {'config_id': 'cfg_e', 'learning_rate': 3e-4, 'dropout': 0.20, 'weight_decay': 1e-4, 'projection_dim': 256, 'hidden_dim': 512, 'epochs': 12, 'patience': 4},
]


def train_or_load_gated_config(config: dict[str, object]) -> dict[str, object]:
    name = f"gated_fusion_e5_clip_{config['config_id']}"
    if not FORCE_RETRAIN and has_complete_experiment(name, model_filename='model.pt'):
        return {'name': name, 'metrics': load_experiment_metrics(name), 'metadata': load_json(experiment_dir(name) / 'metadata.json')}

    train_inputs, val_inputs, test_inputs = make_neural_inputs()
    set_seed(RANDOM_SEED)
    device = resolve_torch_device(DEVICE)
    model = GatedFusionClassifier(
        tabular_dim=train_inputs['tabular'].shape[1],
        text_dim=train_inputs['text'].shape[1],
        image_dim=train_inputs['image'].shape[1],
        projection_dim=int(config['projection_dim']),
        hidden_dim=int(config['hidden_dim']),
        dropout=float(config['dropout']),
    ).to(device)

    train_loader = make_loader(train_inputs, BATCH_SIZE, shuffle=True)
    val_loader = make_loader(val_inputs, BATCH_SIZE, shuffle=False)
    test_loader = make_loader(test_inputs, BATCH_SIZE, shuffle=False)

    positive_count = float(train_inputs['labels'].sum())
    negative_count = float(len(train_inputs['labels']) - positive_count)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([negative_count / max(positive_count, 1.0)], dtype=torch.float32, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(config['learning_rate']), weight_decay=float(config['weight_decay']))

    history = []
    best_pr_auc = float('-inf')
    best_state = None
    best_threshold = 0.5
    epochs_without_improvement = 0
    epoch_iterator = tqdm(range(1, int(config['epochs']) + 1), desc=name) if tqdm is not None else range(1, int(config['epochs']) + 1)

    for epoch in epoch_iterator:
        model.train()
        running_loss = 0.0
        batch_iterator = tqdm(train_loader, desc=f'{name} epoch {epoch}/{config["epochs"]}', leave=False) if tqdm is not None else train_loader
        for tabular, text, image, flags, y_batch in batch_iterator:
            optimizer.zero_grad()
            logits = model(tabular.to(device), text.to(device), image.to(device), flags.to(device))
            loss = criterion(logits, y_batch.to(device))
            loss.backward()
            optimizer.step()
            running_loss += float(loss.item()) * len(y_batch)
            if tqdm is not None:
                batch_iterator.set_postfix(loss=f'{float(loss.item()):.4f}')

        train_loss = running_loss / max(len(train_inputs['labels']), 1)
        val_scores = predict_neural_scores(model, val_loader, device, desc=f'{name} validation {epoch}/{config["epochs"]}')
        val_pr_auc = float(average_precision_score(labels['val'], val_scores))
        val_threshold = choose_threshold_by_f1(labels['val'], val_scores)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_pr_auc': val_pr_auc})

        if val_pr_auc > best_pr_auc:
            best_pr_auc = val_pr_auc
            best_state = {key: value.detach().cpu() for key, value in model.state_dict().items()}
            best_threshold = val_threshold
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if tqdm is not None:
            epoch_iterator.set_postfix(train_loss=f'{train_loss:.4f}', val_pr_auc=f'{val_pr_auc:.4f}', best_pr_auc=f'{best_pr_auc:.4f}')
        if epochs_without_improvement >= int(config['patience']):
            break

    if best_state is None:
        raise RuntimeError(f'{name} did not produce a best checkpoint.')

    model.load_state_dict(best_state)
    val_scores = predict_neural_scores(model, val_loader, device, desc=f'{name} final validation')
    test_scores = predict_neural_scores(model, test_loader, device, desc=f'{name} final test')
    metrics = {
        'validation': compute_metrics(labels['val'], val_scores, best_threshold),
        'test': compute_metrics(labels['test'], test_scores, best_threshold),
    }
    base = ensure_dir(experiment_dir(name))
    torch.save(model.state_dict(), base / 'model.pt')
    with (base / 'training_history.csv').open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=['epoch', 'train_loss', 'val_pr_auc'])
        writer.writeheader()
        writer.writerows(history)
    save_scores(name, val_scores, test_scores)
    metadata = {
        'group': 'gated_grid',
        'model_kind': 'gated_fusion',
        'config': config,
        'tabular_plus_clip_dim': int(train_inputs['tabular'].shape[1]),
        'text_dim': int(train_inputs['text'].shape[1]),
        'image_dim': int(train_inputs['image'].shape[1]),
    }
    save_experiment(name, metadata, metrics)
    return {'name': name, 'metrics': metrics, 'metadata': metadata}


gated_results = []
if RUN_GATED_GRID:
    for config in GATED_GRID:
        result = train_or_load_gated_config(config)
        gated_results.append(result)
        display(pd.DataFrame([compact_metric_row(result['name'], 'gated_grid', result['metrics'])]))
else:
    print('RUN_GATED_GRID is disabled')


## Late Fusion Over Improved Scores

In [ ]:
candidate_results = simple_results + gated_results
candidate_names = [result['name'] for result in candidate_results]


def fit_or_load_late_fusion_best_v2(candidate_names: list[str]) -> dict[str, object]:
    name = 'late_fusion_best_v2'
    if not FORCE_RETRAIN and has_complete_experiment(name) and (experiment_dir(name) / 'fusion_config.json').exists():
        return {'name': name, 'metrics': load_experiment_metrics(name), 'metadata': load_json(experiment_dir(name) / 'metadata.json')}
    if len(candidate_names) < 2:
        raise ValueError('Need at least two candidate models for late fusion.')

    score_payloads = {candidate: load_scores(candidate) for candidate in candidate_names}
    reference = next(iter(score_payloads.values()))
    for candidate, payload in score_payloads.items():
        if not np.array_equal(reference['y_val'], payload['y_val']) or not np.array_equal(reference['y_test'], payload['y_test']):
            raise ValueError(f'Score labels are not aligned for {candidate}.')

    candidates = []
    for left_name, right_name in combinations(candidate_names, 2):
        left = score_payloads[left_name]
        right = score_payloads[right_name]
        for alpha in LATE_FUSION_ALPHAS:
            val_scores = alpha * left['val_scores'] + (1.0 - alpha) * right['val_scores']
            threshold = choose_threshold_by_f1(labels['val'], val_scores)
            val_metrics = compute_metrics(labels['val'], val_scores, threshold)
            candidates.append(
                {
                    'left_experiment': left_name,
                    'right_experiment': right_name,
                    'alpha': float(alpha),
                    'threshold': float(threshold),
                    'validation_pr_auc': float(val_metrics['pr_auc']),
                    'validation_f1': float(val_metrics['f1']),
                }
            )

    best = max(candidates, key=lambda item: (item['validation_pr_auc'], item['validation_f1']))
    left = score_payloads[best['left_experiment']]
    right = score_payloads[best['right_experiment']]
    val_scores = best['alpha'] * left['val_scores'] + (1.0 - best['alpha']) * right['val_scores']
    test_scores = best['alpha'] * left['test_scores'] + (1.0 - best['alpha']) * right['test_scores']
    metrics = {
        'validation': compute_metrics(labels['val'], val_scores, best['threshold']),
        'test': compute_metrics(labels['test'], test_scores, best['threshold']),
    }
    metadata = {
        'group': 'late_fusion',
        'selected_by': 'validation_pr_auc_then_validation_f1',
        'left_experiment': best['left_experiment'],
        'right_experiment': best['right_experiment'],
        'selected_alpha': best['alpha'],
        'selected_threshold': best['threshold'],
        'candidate_count': len(candidates),
    }
    save_scores(name, val_scores, test_scores)
    save_experiment(name, metadata, metrics)
    save_json(experiment_dir(name) / 'fusion_config.json', {'best': best, 'candidates': candidates, **metadata})
    return {'name': name, 'metrics': metrics, 'metadata': metadata}


late_fusion_result = None
if RUN_LATE_FUSION:
    late_fusion_result = fit_or_load_late_fusion_best_v2(candidate_names)
    display(pd.DataFrame([compact_metric_row(late_fusion_result['name'], 'late_fusion', late_fusion_result['metrics'])]))
    print(load_json(experiment_dir(late_fusion_result['name']) / 'fusion_config.json')['best'])
else:
    print('RUN_LATE_FUSION is disabled')


## Validation And Consistency Checks

In [ ]:
all_experiment_names = [result['name'] for result in candidate_results]
if late_fusion_result is not None:
    all_experiment_names.append(late_fusion_result['name'])

score_alignment_report = validate_score_alignment(all_experiment_names)
if not score_alignment_report['aligned']:
    raise ValueError(f'Score alignment failed: {score_alignment_report}')

consistency_report = {
    'split_checks': split_checks,
    'score_alignment': score_alignment_report,
    'feature_shapes': {
        'tabular_without_row_id': {split: list(tabular_features[split].shape) for split in ('train', 'val', 'test')},
        'text_e5': {split: list(text_features[split].shape) for split in ('train', 'val', 'test')},
        'image_efficientnet': {split: list(image_features[split].shape) for split in ('train', 'val', 'test')},
        'clip_consistency': {split: list(clip_consistency_features[split].shape) for split in ('train', 'val', 'test')},
    },
}
save_json(NAMESPACE_DIR / 'consistency_checks.json', consistency_report)
score_alignment_report


## Final Comparison With V1

In [ ]:
rows = []
for result in candidate_results:
    rows.append(compact_metric_row(result['name'], result['metadata']['group'], result['metrics']))
if late_fusion_result is not None:
    rows.append(compact_metric_row(late_fusion_result['name'], late_fusion_result['metadata']['group'], late_fusion_result['metrics']))

v2_results_table = pd.DataFrame(rows).sort_values(['val_pr_auc', 'val_f1'], ascending=[False, False]).reset_index(drop=True)
ensure_dir(RESULTS_CSV_PATH.parent)
v2_results_table.to_csv(RESULTS_CSV_PATH, index=False)
display(v2_results_table)
print(f'V2 results saved to: {RESULTS_CSV_PATH}')

if V1_RESULTS_PATH.exists():
    v1 = pd.read_csv(V1_RESULTS_PATH)
    v1_compact = v1[['experiment', 'group', 'val_pr_auc', 'val_f1', 'test_pr_auc', 'test_f1']].copy()
    v1_compact.insert(0, 'version', 'v1')
    v2_compact = v2_results_table[['experiment', 'group', 'val_pr_auc', 'val_f1', 'test_pr_auc', 'test_f1']].copy()
    v2_compact.insert(0, 'version', 'v2')
    comparison = pd.concat([v1_compact, v2_compact], axis=0, ignore_index=True)
    comparison.to_csv(V1_COMPARISON_PATH, index=False)
    display(comparison.sort_values(['val_pr_auc', 'val_f1'], ascending=[False, False]).head(20))
    print(f'V1/V2 comparison saved to: {V1_COMPARISON_PATH}')
else:
    print(f'V1 results not found: {V1_RESULTS_PATH}')


## Best Model Summary

In [ ]:
best_row = v2_results_table.iloc[0].to_dict()
best_summary = {
    'selected_by': 'validation_pr_auc_then_validation_f1',
    'best_experiment': best_row['experiment'],
    'best_group': best_row['group'],
    'validation_pr_auc': float(best_row['val_pr_auc']),
    'validation_f1': float(best_row['val_f1']),
    'test_pr_auc': float(best_row['test_pr_auc']),
    'test_precision': float(best_row['test_precision']),
    'test_recall': float(best_row['test_recall']),
    'test_f1': float(best_row['test_f1']),
    'threshold': float(best_row['threshold']),
}
save_json(NAMESPACE_DIR / 'best_model_summary.json', best_summary)
best_summary
